'''

CHANGE NAME BEFORE YOU SCREW UP THE FILE

'''

In [ ]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

# --- autoreload ---
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import asyncio
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output

In [ ]:
from fin_insts import (

# --- core instruments ---
    FinancialInstrument,
    MktData,
    Dates,

# --- tradable instruments ---
    Spot,
    Equity,
    Future,
    FutureSpread,

# --- derived instruments ---
    Subscriber,
    Synthetic,
    BestOf,

# --- builders ---
    make_single_leg_fin_insts
)

In [ ]:
# --- feeds ---
from ws_feeds import WSFeedManager

In [ ]:
# --- utils ---
# from other.Graph_Theory import find_all_node_permutations, connect_nodes_with_edges
from other.Output_Methods import create_output
from other.Class_xlWings import xlWings
xlw = xlWings()

In [ ]:
# --- IBKR ---
from ibkr.Class_IBKR_IB import IBKR_IB
#from ibkr.Class_IBKR_TWS import IBKR_TWS
ibkr = IBKR_IB()

In [ ]:
INPUT_WB_NAME  = "2026 Inputs for Apps.xlsx"

####
INPUT_WS_NAME  = 
INPUT_TBL_NAME = 
####

DB_WB_NAME  = "2026 Crypto Products Database.xlsx"
               
FLATTEN_COLS = ['unit_scalar_dict', 
                'mkt_data_dict', 
                'mkt_comm_dict',      
                'unit_data_dict'
        ]

OUTPUT_COLS = [
           #    'time',
               'my_prod_type',
               'my_platform',
               'crypto_currency',
               'quote_currency',
               'my_name',
               'unit_scalar_dict_price',
               'mkt_data_dict_bid_price',
               'mkt_data_dict_ask_price',
           #    'mkt_comm_dict_join_bid',
               'unit_data_dict_bid_price',
               'unit_data_dict_ask_price'
        ]

In [ ]:
async def standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME):

    df = xlw.get_df(INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME, table=True)
    input_dict = df.set_index('Keys')['Values'].to_dict()
    
    wb  = input_dict['input workbook name']
    ws  = input_dict['true/false sheet name']
    tbl = input_dict['true/false table name']
    true_false_df = xlw.get_df(wb, ws, tbl, table=True)
    true_false_df = true_false_df[true_false_df['TRUE/FALSE'] == True]
    
    wb  = DB_WB_NAME
####    
    ws  = input_dict['crypto long name']
    tbl = input_dict['crypto abbrev'] + "_static_data_table"
####
    db_df = xlw.get_df(wb, ws, tbl, table=True)
    
    merged_df = true_false_df.merge(db_df,how='left',on='my_name')

    fin_inst_objs_list = make_single_leg_fin_insts(merged_df)

    return input_dict, fin_inst_objs_list

In [ ]:
async def main():

    input_dict, objs_list = await standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME)
    
    ws_obj_dict = {obj.my_name : obj for obj in objs_list if obj.platform_id != 'IBKR'}
    ws_feed     = WSFeedManager(ws_obj_dict)
        
    ibkr_obj_dict = {}  # need for ibkr_start_streams below
    ibkr_list     = [obj for obj in objs_list if obj.platform_id == 'IBKR']
    if ibkr_list:
        if 'equity' in [obj.my_prod_type for obj in ibkr_list]:
            ratios_df = xlw.get_df('2026 Crypto ETF Ratios.xlsx', 'BTC RATIOS', 'btc_ratios', table=True)
        else:
            ratios_df = None
            
        await ibkr.connect()
        print("IBKR connected:", ibkr.ib.isConnected())
        
        await asyncio.gather(*(ibkr.create_simple_contract(obj) for obj in ibkr_list))
        await asyncia.gather(*(obj.complete_obj(ratios_df) for obj in ibkr_list))
        
#### 
#        insert ibkr BAG instruments here (future_spread, option_spread, option_combo, etc.)
####         
        
        ibkr_obj_dict = {obj.my_name : obj for obj in ibkr_list}

#### 
#    insert synthetic instruments here  
#### 

#### 
#    insert best_of instruments here  
#### 
        
    output_dict = {**ws_obj_dict, **ibkr_obj_dict,
                  #**syn_obj_dict, ** bo_obj_dict
                  }
    
#### 
#    insert trading and analysis scripts here 
####    
 
    # Run all streams concurrently
    tasks = [
        ws_feed.run(),
        create_output(input_dict, output_dict, FLATTEN_COLS, OUTPUT_COLS),
    ]

    if ibkr_obj_dict:
        tasks.append(ibkr.start_streams(ibkr_obj_dict))

    await asyncio.gather(*tasks)

In [ ]:
await main()